# 🚀 머신러닝 실습 : 카드사용 데이터로 사기거래 예측 모델링 (분류 문제)

* 주어진 데이터세트는 2013년 9월 유럽 카드 소지자들의 신용카드 거래 내역을 담고 있습니다.
* 총 284,807건의 거래(creditcard.csv) 데이터를 이용하여 사기거래 예측 모형을 만들어보세요.
* 모델의 성능은 자유롭게 측정해봅니다!


<br/>

---

<br/>
<br/>

# 0. 라이브러리 불러오기

* 라이브러리를 가져와서 과정을 준비합니다

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier


<br/>

---

<br/>
<br/>

# 1. 데이터 불러오기
* 데이터를 가져와서 과정을 준비합시다.
- 데이터 출처 : https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download

In [2]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\githome\hipython_rep


In [3]:
df = pd.read_csv("./data/creditcard.csv")

<br/>

---

<br/>
<br/>

# 2. 데이터 탐색하기
* 데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.

데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.

In [12]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [16]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


<br/>

---

<br/>
<br/>

# 3. 데이터 전처리
* 전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비

In [4]:
X = df
X = X.drop('Time', axis=1)
X = X.drop('Class', axis=1)
y = df[['Class']]

각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.
- 평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [6]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

<br/>

---

<br/>
<br/>

# 5-1. 모델링 - LogisticRegression

* 본격적으로 모델을 선언하고 학습시킵니다.

- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [7]:
lr_clf = LogisticRegression()
lr_clf.fit(X_train_scaled, y_train)
lr_pred = lr_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [8]:
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

[[56861    13]
 [   36    52]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56874
           1       0.80      0.59      0.68        88

    accuracy                           1.00     56962
   macro avg       0.90      0.80      0.84     56962
weighted avg       1.00      1.00      1.00     56962



In [9]:
accuracy_score(y_test, lr_pred)

0.9991397773954567


<br/>

---

<br/>
<br/>

# 5-2. 모델링 - DecisionTreeClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.

- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [11]:
dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train_scaled, y_train)
dt_pred = dt_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [12]:
print(confusion_matrix(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

[[56858    16]
 [   18    70]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56874
           1       0.81      0.80      0.80        88

    accuracy                           1.00     56962
   macro avg       0.91      0.90      0.90     56962
weighted avg       1.00      1.00      1.00     56962



In [13]:
accuracy_score(y_test, dt_pred)

0.999403110845827


<br/>

---

<br/>
<br/>

# 5-3. 모델링 - RandomForestClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.


- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [14]:
rf_clf = RandomForestClassifier(random_state=42, max_depth=8)
rf_clf.fit(X_train_scaled,y_train)
rf_pred = rf_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [15]:
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

[[56869     5]
 [   23    65]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56874
           1       0.93      0.74      0.82        88

    accuracy                           1.00     56962
   macro avg       0.96      0.87      0.91     56962
weighted avg       1.00      1.00      1.00     56962



In [16]:
accuracy_score(y_test, rf_pred)

0.9995084442259752


<br/>

---

<br/>
<br/>

# 5-4. 모델링 - XGBoost

* 본격적으로 모델을 선언하고 학습시킵니다.


- 모델을 선언하여 객체화시킵니다.
- 모델을 학습 데이터에 맞춰 학습시킵니다.
- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [18]:
xgb = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)
evals = [(X_test_scaled, y_test)]
xgb.fit(X_train_scaled, y_train, early_stopping_rounds=40, 
        eval_set=evals, verbose=True)
xgb_pred = xgb.predict(X_test_scaled)

[0]	validation_0-logloss:0.59839
[1]	validation_0-logloss:0.52078
[2]	validation_0-logloss:0.45607
[3]	validation_0-logloss:0.40137
[4]	validation_0-logloss:0.35465
[5]	validation_0-logloss:0.31441
[6]	validation_0-logloss:0.27951
[7]	validation_0-logloss:0.24908
[8]	validation_0-logloss:0.22241
[9]	validation_0-logloss:0.19895
[10]	validation_0-logloss:0.17821
[11]	validation_0-logloss:0.15985
[12]	validation_0-logloss:0.14355
[13]	validation_0-logloss:0.12906
[14]	validation_0-logloss:0.11615
[15]	validation_0-logloss:0.10462
[16]	validation_0-logloss:0.09432
[17]	validation_0-logloss:0.08510
[18]	validation_0-logloss:0.07683
[19]	validation_0-logloss:0.06942
[20]	validation_0-logloss:0.06278
[21]	validation_0-logloss:0.05681
[22]	validation_0-logloss:0.05144
[23]	validation_0-logloss:0.04662
[24]	validation_0-logloss:0.04227
[25]	validation_0-logloss:0.03836
[26]	validation_0-logloss:0.03483
[27]	validation_0-logloss:0.03165
[28]	validation_0-logloss:0.02879
[29]	validation_0-loglos

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [19]:
print(confusion_matrix(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

[[56869     5]
 [   18    70]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56874
           1       0.93      0.80      0.86        88

    accuracy                           1.00     56962
   macro avg       0.97      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



In [20]:
accuracy_score(y_test, xgb_pred)

0.9995962220427653

<br/>

---


<br/>

## 7.  위 4가지 모델의 학습 & 예측 & 평가 결과를 확인하고 최고 성능을 내는 모델을 찾아봅시다!

- 어떤 모델이 가장 성능이 좋은가요 ?

4 모델 전부 99.9퍼센트 이상의 정확도를 보였지만 XGBoost가 미세하게 조금 더 높았다